# Prep

## Import stuff

In [3]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math
import ast

## Some magical magic to make the R stuff work

In [ ]:
# The rpy2/R session and CFA machinery now live in the hitop_cfa package.
# Importing it starts the embedded R session: it sets the BLAS threading
# env vars (if unset) and loads base, utils, lavaan, and the patched semTools.
import hitop_cfa
from hitop_cfa.r_env import (ro, rbase, utils, lavaan, semtools,
                             RRuntimeError, pandas2ri)

import rpy2.ipython.html
rpy2.ipython.html.init_printing()

# Paths

In [6]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'manual_temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'


## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [ ]:
import platform


def get_architecture():
    # Get the raw machine architecture string
    arch = platform.machine().lower()
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    elif "x86" in arch or "amd" in arch or "i386" in arch or "i686" in arch:
        return "x86"
    else:
        return f"Unknown ({arch})"


total_cpus = os.cpu_count()
# account for hyperthreading
arch = get_architecture()
if arch == 'ARM':
    cpus_to_use = total_cpus - 2
else:
    cpus_to_use = total_cpus // 2 - 1
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000

## SET SEEDS !!!!!!!!!!

In [8]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [9]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x1466d7510> [0]

### TEST THE SEEDS!!!!!!!!

In [10]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [11]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.3650586371545244,-1.57091128601566,1.1419085835874878


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [ ]:
from hitop_cfa import (
    build_luts,
    check_hitop_ids,
    cfa_helper_func,
    run_specific_cfa,
    do_three_way_cfa_stepwise_mi,
    do_three_way_cfa_stepwise_scalar,
    exhaustive_cfa_ablations,
    set_seeds,
    silence_r,
)

# item-text lookups (was load_item_lookup + inline lut construction)
_luts = build_luts(path_to_item_lookup, path_to_cogmood_questions)
item_lookup = _luts['item_lookup']
item_lut = _luts['item_lut']
phq_lut = _luts['phq_lut']
gad_lut = _luts['gad_lut']
baars_lut = _luts['baars_lut']

## CFA wrapper functions

# Run Main Code

## Load preprocessed data and concatinate

In [ ]:
# load
data_val = pd.read_csv(path_save_val)
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# concatinate
data_val_genpop = pd.concat([data_val, data_gp])
data_val_enriched = pd.concat([data_val, data_en])
data_genpop_enriched = pd.concat([data_gp, data_en])

# Dataset pairs for the hitop_cfa functions. Unified on the full
# genpop+enriched source (the old norecontact variant was column-identical
# on every CFA item, so this cannot change results).
datasets = {'val_gp': data_val_genpop, 'val_en': data_val_enriched, 'gp_en': data_genpop_enriched}
datasets_gp_en = {'gp_en': data_genpop_enriched}

# Do main CFA analysis

## 3-way CFA


### 1. Check original items
Get full 3-way CFA results (config, metric, scalar, strict) for all original scales.

In [24]:
orig_items = {
    'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
    'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop240 + hitop248 + hitop265',
    'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
    'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
    'cognitive_problems': 'cognitive_problems =~hitop67 + hitop159 + hitop189 + hitop142',
    'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
    'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
    'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
    'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
    'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
    'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
    'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
    'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'
}

In [ ]:
with open(log_dir / "mylog_3wayCFA_origscales_seed12345_v2.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in orig_items.items():
            # print which scale we are processing through R - this way it doesn't get saved in the log file
            ro.globalenv['scale_to_print'] = scale
            ro.r('print(scale_to_print)')
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test
            run_specific_cfa(
                whichscale=scale,
                item_list=items_list,
                whichcfa='strict',
                datasets=datasets,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
            )

### 2. Find invariant subsets

#### Looping through all scales where full scales did not pass metric invariance:

In [ ]:
lookingforinv_log = log_dir / 'mylog_3wayCFA_lookingforinv_seed12345_v1.txt'
noninvariant_scales = [
    'anhedonic_depression',
    'anxious_worry',
    'appetite_gain',
    'cognitive_problems',
    'hyposomnia',
    'panic',
    'separation_insecurity',
    'shame_guilt',
    'situational_phobia',
    'social_anxiety',
    'well_being'
]
with lookingforinv_log.open('w') as f:
    with redirect_stdout(f):
        for scale in noninvariant_scales:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = orig_items[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = exhaustive_cfa_ablations(
                    whichscale=scale,
                    whichcfa='metric',
                    howmanyitems=how_many_to_try,
                    orig_items=orig_items,
                    datasets=datasets,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                )
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale:
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

In [34]:
lfi_out = lookingforinv_log.read_text().split('\n')



In [62]:
lfi_parsing = []
for lix, line in enumerate(lfi_out):
    if line.startswith('!!!!!'):
        row= dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        lfi_parsing.append(row)


In [ ]:
ds_pairs = ['val_gp', 'val_en', 'gp_en']
inv_levels = ['config', 'metric', 'scalar', 'strict']

In [195]:
inv_dat = []
for lprow in lfi_parsing:
    scale_stats = ast.literal_eval(lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        ogformula = orig_items[lprow['scale']]
        itemformula_only = ogformula.split("=~",1)[1]
        ogitems = itemformula_only.split(" + ")
        row = lprow.copy()
        item_nos = iss[0]
        items = [item_lut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [item_lut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        inv_dat.append(row)
inv_dat = pd.DataFrame(inv_dat)

In [196]:
review_notes = [
    {
        'scale': 'anhedonic_depression',
        'ssix': 2,
        'note': '2 energy items and one anhedonic item exluded',
        'stepwise_agrees':True
    },
    {
        'scale': 'anxious_worry',
        'ssix': 1,
        'note': 'excluded items are more somatic',
    },
    {
        'scale': 'appetite_gain',
        'ssix': 0,
        'note': 'this solution exludes the only item that refers to a thought',
        'stepwise_agrees':True
    },
    {
        'scale': 'cognitive_problems',
        'note': "failed in both stepwise and exhaustive because it's only 3 items",
    },
    {
        'scale': 'hyposomnia',
        'ssix': 0, 
        'note': "stepwise solution found by exhaustive, can't interpret either solution",
        'stepwise_agrees': True
    },
    {
        'scale': 'panic',
        'ssix': 2,
        'note': "scale is just the the physical symptoms of panic attack, arbitrary to pick one, but nauseated is perhaps the symptom most likely to be caused by something else",
        'stepwise_agrees': False
    },
    {
        'scale': 'shame_guilt',
        'ssix': 0 ,
        'note': 'may indicate that the scale is misnamed',
        'stepwise_agrees': True
    },
    {
        'scale': 'separation_insecurity', 
        'ssix': 0,
        'note': "could not handle rejection is an ambiguous item, what does failing to handle mean here",
        'stepwise_agrees': False
    },
    {
        'scale': 'situational_phobia',
        'ssix': 1,
        'note': 'avoiding riding in elevators is not writen as "I was afraid"',
        'stepwise_agrees': False
    },
    {
        'scale': 'social_anxiety', 
        'ssix': 0,
        'note': "stepwise bug, not a great interpretation"
    },
    {
        'scale': 'well_being',
        'ssix': 0,
        'note': "agrees with both exhaustive and stepwise procedures, removed items are actions, retained are all 'I felt'",
        'stepwise_agrees': False
    }
]
review_notes = pd.DataFrame(review_notes)
review_notes['exhaustive_choice'] = True
review_notes.loc[review_notes.scale == 'cognitive_problems', 'exhaustive_choice'] = False

In [197]:
review_notes

,scale,ssix,note,stepwise_agrees,exhaustive_choice
0,anhedonic_depression,2.0,2 energy items and one anhedonic item exluded,True,True
1,anxious_worry,1.0,excluded items are more somatic,NaN,True
2,appetite_gain,0.0,this solution exludes the only item that refers to a thought,True,True
3,cognitive_problems,NaN,failed in both stepwise and exhaustive because it's only 3 items,NaN,False
4,hyposomnia,0.0,"stepwise solution found by exhaustive, can't interpret either solution",True,True
5,panic,2.0,"scale is just the the physical symptoms of panic attack, arbitrary to pick one, but nauseated is...",False,True
6,shame_guilt,0.0,may indicate that the scale is misnamed,True,True
7,separation_insecurity,0.0,"could not handle rejection is an ambiguous item, what does failing to handle mean here",False,True
8,situational_phobia,1.0,"avoiding riding in elevators is not writen as ""I was afraid""",False,True
9,social_anxiety,0.0,"stepwise bug, not a great interpretation",NaN,True


In [198]:
inv_dat = inv_dat.merge(review_notes, how='outer', on=['scale', 'ssix'])
inv_dat['exhaustive_choice'] = inv_dat.exhaustive_choice.fillna(False)

In [199]:
inv_dat.to_pickle(cfa_dir / 'exhaustive.pkl')

In [126]:
for row in inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for ii in row.removed:
        print(ii)
    print()
    print()

########################
Scale: anhedonic_depression, Subset_id: 0
########################
It felt like there wasn’t anything interesting or fun to do.
I didn’t look forward to seeing friends or family.
I felt depressed.
Nothing made me laugh.
I was unable to enjoy things like I normally do.
I felt emotionally numb.
I was a lot less talkative than usual.
---------REMOVED---------------
It took a lot of effort to do everyday activities.
Nothing seemed interesting to me.
I had very little energy.


########################
Scale: anhedonic_depression, Subset_id: 1
########################
It felt like there wasn’t anything interesting or fun to do.
I felt depressed.
Nothing seemed interesting to me.
Nothing made me laugh.
I was unable to enjoy things like I normally do.
I felt emotionally numb.
I was a lot less talkative than usual.
---------REMOVED---------------
I didn’t look forward to seeing friends or family.
It took a lot of effort to do everyday activities.
I had very little ener

### 3. (if needed) Check invariant subsets

In [ ]:
##  all inv scales
to_test_inv = {'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop93 + hitop182 + hitop230 + hitop246',
         'anxious_worry': 'anxious_worry =~hitop34 + hitop89 + hitop203 + hitop248',
         'appetite_gain': 'appetite_gain =~hitop120 + hitop243 + hitop275',
         'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
         'cognitive_problems': 'cognitive_problems =~hitop67 + hitop189 + hitop142',
         'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
         'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
         'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
         'separation_insecurity': 'separation_insecurity =~hitop50 + hitop69 + hitop81 + hitop136 + hitop151 + hitop197',
         'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
         'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247',
         'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop129 + hitop204 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop149 + hitop200 + hitop244 + hitop250 + hitop281'}

with open(log_dir / "mylog_3wayCFA_invsubsets_seed12345_2.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test
            run_specific_cfa(
                whichscale=scale,
                item_list=items_list,
                whichcfa='strict',
                datasets=datasets,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
            )

## For interpretation - check what items mean

In [32]:
scales = {
    'items_anhedonic_depression':['39','77','84','92','93','123','157','182','230','246'],
    'items_anxious_worry': ['20','34','89','203','248','265'],
    'items_appetite_gain': ['120','141','243','275'],
    'items_appetite_loss': ['280','283','109'],
    'items_cognitive_problems': ['67','189','142'],
    'items_hyposomnia': ['99','181','5','66','231'],
    'items_insomnia': ['160','254','261','268'],
    'items_panic': ['15','104','126','211','215','257'],
    'items_separation_insecurity': ['40','50','69','81','113','136','151','197'],
    'items_shame_guilt': ['72','140','143','220'],
    'items_situational_phobia': ['16','165','225','247','278'],
    'items_social_anxiety': ['1','17','114','117','124','129','204','222','236','258'],
    'items_well_being': ['9','23','54','106','149','200','244','245','250','281']}

for scale, items in scales.items():
    print(scale)
    check_hitop_ids(items, my_item_lookup=item_lookup)

items_anhedonic_depression
   hitop39   It felt like there wasn’t anything interesting or fun to do.
   hitop77   I didn’t look forward to seeing friends or family.
   hitop84   I felt depressed.
   hitop92   It took a lot of effort to do everyday activities.
   hitop93   Nothing seemed interesting to me.
   hitop123   Nothing made me laugh.
   hitop157   I had very little energy.
   hitop182   I was unable to enjoy things like I normally do.
   hitop230   I felt emotionally numb.
   hitop246   I was a lot less talkative than usual.


items_anxious_worry
   hitop20   I felt tense.
   hitop34   Thoughts were racing through my head.
   hitop89   I had a lot of nervous energy.
   hitop203   I felt very stressed.
   hitop248   I worried about almost everything.
   hitop265   I was overwhelmed by anxiety.


items_appetite_gain
   hitop120   I could not keep myself from eating.
   hitop141   I thought a lot about food.
   hitop243   I stuffed myself with food.
   hitop275   I ate even when I

# Analysis for BAARS, GAD, PHQ

In [ ]:
other_scales = {
    'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
    'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
    'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
    'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
    'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
    'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}

other_log = log_dir /'mylog_CFA_baarsgadphq_seed12345.txt'
with other_log.open("w") as f:
    with redirect_stdout(f):
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")

            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")

            # +++ GENPOP VS ENRICHED +++
            print('\n -----> GENPOP VS ENRICHED <----- ')
            flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(
                            scalename=scale,
                            list_of_items=items_list,
                            do_metric=True,
                            do_scalar=True,
                            do_strict=True,
                            mydata_python=data_genpop_enriched,
                            mydata_temp_path=path_to_helpfile,
                            num_iter=num_iter,
                            cpus_to_use=cpus_to_use)

In [ ]:
other_exhaustive_logs = log_dir /'mylog_exhuastive_inv_search_baarsgadphq_seed12345.txt'

with other_exhaustive_logs.open("w") as f:
    with redirect_stdout(f):
        for scale in ['phq_sum', 'gad_sum', 'baars_inattention_sum', 'baars_sct_sum']:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = other_scales[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = exhaustive_cfa_ablations(
                    whichscale=scale,
                    whichcfa='metric',
                    howmanyitems=how_many_to_try,
                    orig_items=other_scales,
                    datasets=datasets_gp_en,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                )
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale:
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

In [ ]:
other_lfi_out = other_exhaustive_logs.read_text().split('\n')
other_lfi_parsing = []
for lix, line in enumerate(other_lfi_out):
    if line.startswith('!!!!!'):
        row = dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        other_lfi_parsing.append(row)
other_ds_pairs = ['gp_en']

In [ ]:
other_inv_dat = []
for lprow, olut in zip(other_lfi_parsing, [phq_lut, gad_lut, baars_lut['inattention'], baars_lut['sct']]):
    scale_stats = ast.literal_eval(other_lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        row = lprow.copy()
        item_nos = iss[0]
        ogitems = list(olut.keys())
        items = [olut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [olut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in other_ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        other_inv_dat.append(row)
other_inv_dat = pd.DataFrame(other_inv_dat)

In [ ]:
for row in other_inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for ii in row.removed:
        print(ii)
    print()
    print()

In [ ]:
# check baars sum although we don't report it:
other_scales = {'baars_SUM_no_sct': 'baars_SUM_no_sct =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9 + hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5 + impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
               'baars_SUM_with_sct': 'baars_SUM_with_sct =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9 + hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5 + impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4 + sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}

with open(log_dir / "mylog_CFA_baarsgadphq_seed12345.txt", "a") as f:
    with redirect_stdout(f):
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")

            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")

            # +++ GENPOP VS ENRICHED +++
            print('\n -----> GENPOP VS ENRICHED <----- ')
            flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(
                            scalename=scale,
                            list_of_items=items_list,
                            do_metric=True,
                            do_scalar=True,
                            do_strict=True,
                            mydata_python=data_genpop_enriched,
                            mydata_temp_path=path_to_helpfile,
                            num_iter=num_iter,
                            cpus_to_use=cpus_to_use)